# Employee Dataset Analysis Project


### 1. Load Dataset


In [1]:
import pandas as pd
import numpy as np
import os

path = "../csv/employees_100.csv" if os.path.exists("../csv/employees_100.csv") else "csv/employees_100.csv"
df = pd.read_csv(path)

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:\n", df.head())


Dataset Shape: (100, 10)

First 5 rows:
    employee_id            name  ... projects_completed joining_date
0          184    Swati Kapoor  ...                  6   2021-07-02
1          154  Kavya Malhotra  ...                  8   2024-02-05
2          171    Manish Desai  ...                 18   2019-04-24
3          146        Neha Das  ...                  1   2022-12-13
4          145       Isha Nair  ...                 14   2019-11-25

[5 rows x 10 columns]


### 2. Inspect Data & Check Missing Values


In [2]:
print("Dataset Info:")
df.info()

print("\nMissing values count:\n", df.isnull().sum())
print("\nBasic Statistics:\n", df.describe())


Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   employee_id         100 non-null    int64  
 1   name                100 non-null    str    
 2   department          92 non-null     str    
 3   city                90 non-null     str    
 4   age                 94 non-null     float64
 5   salary              92 non-null     str    
 6   experience          100 non-null    int64  
 7   performance_score   100 non-null    int64  
 8   projects_completed  100 non-null    int64  
 9   joining_date        100 non-null    str    
dtypes: float64(1), int64(4), str(5)
memory usage: 7.9 KB

Missing values count:
 employee_id            0
name                   0
department             8
city                  10
age                    6
salary                 8
experience             0
performance_score      0
projects_completed     

### 3. Data Cleaning


In [3]:
# Clean salary column (remove commas and convert to numeric)
df['salary'] = df['salary'].astype(str).str.replace(',', '').str.strip()
df['salary'] = pd.to_numeric(df['salary'], errors='coerce')
df['salary'] = df['salary'].fillna(df['salary'].median())

# Clean age column
df['age'] = df['age'].fillna(df['age'].median())

# Clean department and city strings
df['department'] = df['department'].fillna('Other').astype(str).str.strip().str.title()
df['city'] = df['city'].fillna('Unknown').astype(str).str.strip().str.title()

print("Missing values after cleaning:\n", df.isnull().sum())
print("\nCleaned salary summary:")
print(f"Min: {df['salary'].min()}, Max: {df['salary'].max()}, Mean: {df['salary'].mean():.2f}")


Missing values after cleaning:
 employee_id           0
name                  0
department            0
city                  0
age                   0
salary                0
experience            0
performance_score     0
projects_completed    0
joining_date          0
dtype: int64

Cleaned salary summary:
Min: 45835.0, Max: 88366.0, Mean: 65090.75


### 4. Department-wise Salary & Headcount


In [4]:
dept_summary = df.groupby('department')['salary'].agg(
    headcount='count',
    avg_salary='mean',
    min_salary='min',
    max_salary='max'
).round(2)

print("Department Performance Summary:\n", dept_summary)


Department Performance Summary:
               headcount  avg_salary  min_salary  max_salary
department                                                 
Data Science         10    76450.80     69765.0     88216.0
Engineering          17    74038.21     62693.0     84870.0
Finance              18    65811.83     58374.0     80597.0
Hr                   19    55756.05     45835.0     66363.0
Marketing            19    59382.00     49693.0     69499.0
Other                 8    72510.00     63209.0     88366.0
Sales                 9    59289.06     52404.0     68490.0


### 5. Top Earners & Top Performers


In [5]:
# Top 5 highest salaries
top_earners = df.sort_values(by='salary', ascending=False)[['name', 'department', 'city', 'salary']].head(5)
print("Top 5 Earners:\n", top_earners)

# High performers (score >= 8)
top_performers = df[df['performance_score'] >= 8][['name', 'department', 'performance_score', 'salary']]
print(f"\nTotal High Performers (Score >= 8): {len(top_performers)}")
print(top_performers.head(5))


Top 5 Earners:
               name    department       city   salary
74  Tanvi Malhotra         Other     Mumbai  88366.0
75  Tanvi Malhotra         Other     Mumbai  88366.0
4        Isha Nair  Data Science    Unknown  88216.0
88    Deepak Verma  Data Science  Hyderabad  86380.0
84     Rahul Menon   Engineering  Hyderabad  84870.0

Total High Performers (Score >= 8): 14
              name    department  performance_score   salary
1   Kavya Malhotra   Engineering                  8  77058.0
3         Neha Das   Engineering                  8  69993.0
4        Isha Nair  Data Science                  8  88216.0
5      Rahul Gupta         Sales                  9  52404.0
9  Ananya Malhotra   Engineering                  8  68591.0


### 6. City Distribution & Experience


In [6]:
print("Employees per City:\n", df['city'].value_counts())

print("\nAverage experience per department:\n", df.groupby('department')['experience'].mean().round(1))


Employees per City:
 city
Hyderabad    19
Pune         14
Delhi        14
Kolkata      13
Mumbai       10
Unknown      10
Bangalore    10
Chennai      10
Name: count, dtype: int64

Average experience per department:
 department
Data Science     3.8
Engineering      4.7
Finance          4.9
Hr               3.8
Marketing        4.0
Other           10.4
Sales            4.9
Name: experience, dtype: float64


### 7. Feature Engineering (Bonus & Seniority)


In [7]:
# Bonus based on performance score percentage
df['bonus'] = (df['salary'] * (df['performance_score'] / 100)).round(2)
df['total_compensation'] = (df['salary'] + df['bonus']).round(2)

# Seniority level based on experience
df['seniority'] = np.where(df['experience'] >= 5, 'Senior', 'Junior')

print("Sample after feature engineering:\n", df[['name', 'department', 'experience', 'seniority', 'salary', 'bonus', 'total_compensation']].head(5))


Sample after feature engineering:
              name    department  ...    bonus total_compensation
0    Swati Kapoor  Data Science  ...  3535.95           74254.95
1  Kavya Malhotra   Engineering  ...  6164.64           83222.64
2    Manish Desai  Data Science  ...  4194.66           74105.66
3        Neha Das   Engineering  ...  5599.44           75592.44
4       Isha Nair  Data Science  ...  7057.28           95273.28

[5 rows x 7 columns]


### 8. Export Cleaned Data


In [8]:
out_dir = "../csv" if os.path.exists("../csv") else "csv"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "cleaned_employees_summary.csv")

df.to_csv(out_path, index=False)
print(f"Cleaned dataset saved successfully to: {out_path}")


Cleaned dataset saved successfully to: csv\cleaned_employees_summary.csv
